In [ ]:

using System;
using System.Diagnostics;
using System.Threading.Tasks;

Console.WriteLine("Building Chronos.sln (Debug)...");
var p = new Process();
p.StartInfo = new ProcessStartInfo {
    FileName = "dotnet",
    Arguments = "build Chronos.sln -c Debug",
    UseShellExecute = false,
    RedirectStandardOutput = true,
    RedirectStandardError = true
};
p.Start();
var stdout = p.StandardOutput.ReadToEnd();
var stderr = p.StandardError.ReadToEnd();
p.WaitForExit();
Console.WriteLine(stdout);
if (!string.IsNullOrWhiteSpace(stderr)) Console.WriteLine(stderr);

In [2]:
// Load dependencies explicitly for Dotnet Interactive.
#r "nuget: YamlDotNet, 16.3.0"
#r "nuget: Docker.DotNet, 3.125.15"
#r "nuget: Polly, 8.6.6"

#r "D:/repos/Chronos/src/Chronos.Core/bin/Debug/net8.0/Chronos.Core.dll"

using Chronos.Core;

Console.WriteLine("Chronos.Core loaded.");

Installed Packages Docker.DotNet, 3.125.15 Polly, 8.6.6 YamlDotNet, 16.3.0

Chronos.Core loaded.


In [3]:
#r "D:\repos\Chronos\src\Chronos.Core\bin\Debug\net8.0/Chronos.Core.dll"

In [6]:


var compose = new ComposeBuilder()
  .WithVersion("3.8")
  .WithProjectName("pg-nginx")
  .AddNetwork("app-net")
  .AddVolume("pgdata")
    .AddService(s => s
        .WithName("postgres")
        .UseImage("postgres:16-alpine")
        .WithRestartPolicy("unless-stopped")
        .AddEnvironment("POSTGRES_USER", "app")
        .AddEnvironment("POSTGRES_PASSWORD", "app_password")
        .AddEnvironment("POSTGRES_DB", "appdb")
        .AddPort(5432, 5432)
        .ConnectToNetwork("app-net")
        .AddVolume("pgdata", "/var/lib/postgresql/data")
    )
    .AddService(s => s
        .WithName("nginx")
        .UseImage("nginx:alpine")
        .WithRestartPolicy("unless-stopped")
        .AddPort(8080, 80)
        .ConnectToNetwork("app-net")
        .DependsOn("postgres")
    );

Console.WriteLine("Compose builder prepared.");


Compose builder prepared.


In [ ]:
Console.WriteLine(compose.Display());

In [5]:
await compose.ValidateAsync()

IsValid,True
Errors,(empty)
Warnings,(empty)


In [7]:
await compose.TestAsync();

[Test] Validating compose...
[Test] Running local test (project 'pg-nginx')...
🧪 docker-compose up -d (local test)...
[docker-compose cmd] docker-compose -f "docker-compose.yml" -p pg-nginx up -d
🕒 Waiting for compose containers (timeout: 00:01:00)...

✅ pg-nginx-nginx-1: running
✅ pg-nginx-postgres-1: running
✅ Test passed.
[docker-compose cmd] docker-compose -f "docker-compose.yml" -p pg-nginx down -v


In [8]:
await compose.StartAsync();

[Start] Writing compose to 'docker-compose.yml'...
[Start] docker-compose up -d (project 'pg-nginx')...
🧪 docker-compose up -d (local test)...
[docker-compose cmd] docker-compose -f "docker-compose.yml" -p pg-nginx up -d
🕒 Waiting for compose containers (timeout: 00:01:00)...

✅ pg-nginx-nginx-1: running
✅ pg-nginx-postgres-1: running
✅ Test passed.


In [ ]:
await compose.StopAsync()

In [ ]:
compose.ToString()

In [ ]:
compose.Display()

In [ ]:
var apiUrl = "https://localhost:7004";
await ComposeBuilder.ListRemoteProjectsAsync(apiUrl)

In [ ]:
var com = await ComposeBuilder.LoadFromAgentAsync(apiUrl,"chronos");

In [ ]:
com.ToFluentApiCode()

In [ ]:
await com.StartAsync()

In [ ]:
await com.StopAsync()

In [ ]:
com.WithProjectName("hello");

com.ToFluentApiCode()

In [ ]:
await com.StartAsync()

In [ ]:
await com.PublishAsync(apiUrl)

In [ ]:
await ComposeBuilder.ListRemoteProjectsAsync(apiUrl)

In [ ]:
await com.StartRemoteAsync(apiUrl)